# FORGE M7 — Larger-model pilot: 125M-class on real code (FP16, single T4)

**Purpose:** the last step before any full-run decision. Train the
125M-class model (d=768, L=18 — 111.7M params with the byte vocab) on a
real codeparrot-clean slice, measure throughput AT 125M scale (no
FLOPs-ratio extrapolation), prove kill/resume at scale, and validate the
train → checkpoint → eval pipeline end to end.

**Budget (GPU, single T4):** probe ~5 min + kill test ~5 min + main run
~60–90 min + eval ~15 min ≈ **under 2 hours**, inside one 12 h session
and a small fraction of the ~30 GPU-h/week quota. Every phase checkpoints,
so a session interruption costs minutes, not the run — the explicit
resume cell below exists for exactly that case.

**What this does NOT prove:** 50M tokens is ~2% of the 2.5B plan. A pilot
score near zero on held-out coding tasks is EXPECTED and is not a failure
of the pilot — the pilot validates the pipeline and the scaling numbers.
Capability claims await the full run. The held-out split (12 tasks) is
NEVER touched here; eval uses the dev split (12 tasks) only.

In [ ]:
import torch, subprocess, json, os, sys, shutil
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (torch):", torch.version.cuda if torch.cuda.is_available() else None)
n = torch.cuda.device_count()
print("visible GPUs:", n)
for i in range(n):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"  gpu{i}: {props.name} | total VRAM {total/1e9:.1f} GB | free {free/1e9:.1f} GB")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
      capture_output=True, text=True).stdout)
print("disk free GB:", shutil.disk_usage('/kaggle/working').free / 1e9)
assert torch.cuda.is_available(), "NO CUDA — enable the GPU accelerator in notebook settings first"
print("ENV CHECK: CUDA is usable")

## Get the code

Same pattern as EXP-005/006: clone from GitHub (pre-filled) and pin to an
exact commit. The pin is filled in before handover — never a moving branch.

In [ ]:
import os, shutil, subprocess, zipfile

def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('command failed: ' + ' '.join(cmd) + '\n' + r.stderr[-2000:])
    return r.stdout.strip()

REPO_URL = "https://github.com/mohnish-ydv/forge"
FORGE_COMMIT = "FILL_AFTER_PUSH"  # pinned GitHub commit — never a moving branch
ZIP_PATH = ""

REPO_DIR = '/kaggle/working/forge'
if REPO_URL:
    if not os.path.isdir(REPO_DIR):
        sh(['git', 'clone', REPO_URL, REPO_DIR])
        print('cloned', REPO_URL)
    else:
        print('repo dir already exists — reusing it')
elif ZIP_PATH:
    zipfile.ZipFile(ZIP_PATH).extractall(REPO_DIR)
    if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
        subs = [d for d in os.listdir(REPO_DIR) if os.path.isdir(os.path.join(REPO_DIR, d))]
        if len(subs) == 1 and os.path.isdir(os.path.join(REPO_DIR, subs[0], 'src')):
            for item in os.listdir(os.path.join(REPO_DIR, subs[0], item)):
                shutil.move(os.path.join(REPO_DIR, subs[0], item), os.path.join(REPO_DIR, item))
            os.rmdir(os.path.join(REPO_DIR, subs[0]))
    print('unzipped', ZIP_PATH)
else:
    raise SystemExit('Set REPO_URL or ZIP_PATH above, then re-run')
print('code dir:', REPO_DIR)

In [ ]:
os.chdir('/kaggle/working/forge')
actual = None
if REPO_URL:
    sh(['git', 'fetch', 'origin', FORGE_COMMIT])
    sh(['git', 'checkout', '--force', FORGE_COMMIT])
    sh(['git', 'clean', '-fdx', '-e', 'checkpoints'])
    actual = sh(['git', 'rev-parse', 'HEAD'])
    assert actual.startswith(FORGE_COMMIT), f'pinned {FORGE_COMMIT} but HEAD is {actual}'
    print('pinned commit verified:', actual)
else:
    print('ZIP path: commit pinning not verifiable — record your repo state manually')
print('ACTUAL_HEAD=' + str(actual))

## Corpus prep — real code, same filters as the audit

Shard **2** (not shard 1 — shard 1 is the EXP-007 audit slice and stays
pristine for measurement). First 45,000 rows → the exact M2 pipeline
(license → quality → secrets → exact dedup → near-dup → HumanEval+
decontamination) → concatenated bytes → `data/pilot/train.bin`.
Expected yield at audit rates (Stage B measured: 46.0% retained,
~8,958 B/file): ~20,700 files, ~186 MB, ~46M tokens (bytes/4 estimate).
6,100 steps × 8,192 tokens = 50M tokens processed, i.e. just over one
epoch over the pilot slice. The cell also verifies heldout-task isolation
(prompt+tests vs retained, same check M4 ran) and that no retained doc
carries secret hits. Manifest-style stats are saved; file contents are
never printed.

In [ ]:
import gzip, json, os, sys, collections, urllib.request
sys.path.insert(0, 'src')
from data_pipeline import PipelineConfig, new_record, run_pipeline
from decontam import Decontaminator, load_fingerprints, normalize_code
import hashlib

REV = "35a59fb025bc0a102f7d96eac09d145b896d487b"
SHARD_URL = f"https://huggingface.co/datasets/codeparrot/codeparrot-clean/resolve/{REV}/file-000000000002.json.gz"
SHARD_GZ = "/kaggle/working/shard2.json.gz"
N_ROWS = 45000
if not os.path.exists(SHARD_GZ):
    print('downloading shard 2 (~247 MB)...')
    urllib.request.urlretrieve(SHARD_URL, SHARD_GZ)
print('shard size MB:', round(os.path.getsize(SHARD_GZ) / 1e6, 1))

records = []
with gzip.open(SHARD_GZ, 'rt') as f:
    for i, line in enumerate(f):
        if i >= N_ROWS:
            break
        row = json.loads(line)
        records.append(new_record(
            i, row.get('content', '') or '', path=row.get('path', '') or '',
            repo=row.get('repo_name', '') or '', language='python',
            license=row.get('license', '') or '',
            source='codeparrot/codeparrot-clean', source_rev=REV))
print(f'rows in: {len(records)}')
fp = load_fingerprints('benchmarks/fingerprints.json')
dec = Decontaminator(fp)
all_recs, kept, stats = run_pipeline(records, PipelineConfig(max_files=len(records)), decontam=dec)
kept_recs = [r for r in all_recs if not r['excluded']]
secret_kept = sum(1 for r in kept_recs if r.get('secret_hits'))
assert secret_kept == 0, 'retained docs with secret hits!'
# heldout-task isolation (prompt+tests, NOT quarantined solutions)
bench = json.load(open('benchmarks/forge_heldout_v1.json'))
corpus_hashes = {r['sha256'] for r in kept_recs}
hits = []
for t in bench['tasks']:
    text = t['prompt'] + '\n' + '\n'.join(t['tests'])
    if hashlib.sha256(text.encode()).hexdigest() in corpus_hashes or \
       ('N' + hashlib.sha256(normalize_code(text).encode()).hexdigest()) in corpus_hashes:
        hits.append(t['id'])
assert not hits, f'heldout isolation breach: {hits}'
# write train.bin: concatenated retained content bytes (byte-level model)
os.makedirs('data/pilot', exist_ok=True)
blob = b''.join(r['content'].encode('utf-8') for r in kept)
with open('data/pilot/train.bin', 'wb') as f:
    f.write(blob)
lic = collections.Counter(r['license'].lower() for r in kept_recs)
corpus_stats = {
    'shard': 'file-000000000002.json.gz', 'source_rev': REV, 'rows_in': len(records),
    'retained': len(kept_recs), 'retained_bytes': stats['bytes_retained'],
    'tokens_est_bytes_div_4': stats['bytes_retained'] / 4,
    'licenses_kept': dict(lic.most_common()),
    'excluded_license': stats.get('excluded_license'),
    'excluded_unparseable': stats.get('excluded_unparseable'),
    'excluded_secret': stats.get('excluded_secret'),
    'near_dups': stats.get('near_dups'),
    'secret_kept': secret_kept, 'heldout_task_hits': hits,
}
json.dump(corpus_stats, open('/kaggle/working/corpus_stats.json', 'w'), indent=2)
print('retained:', len(kept_recs), '| bytes:', stats['bytes_retained'],
      '| tokens est:', round(stats['bytes_retained'] / 4 / 1e6, 1), 'M')
print('licenses:', dict(lic.most_common()))
print('train.bin bytes:', len(blob))

## Phase A — throughput probe at 125M scale (300 steps, FP16)

125M-class config (d=768, L=18, 12 heads / 3 KV) on the pilot data.
Same batch/seq as the M6 benchmark (16 × 512). T_max=6100 (the FULL
schedule) so the main run can resume from this checkpoint without a
schedule change. The probe's `steady_tok_per_s` IS the 125M throughput
number — no FLOPs-ratio extrapolation.

In [ ]:
import subprocess, json, os

PILOT = ['--device', 'cuda', '--precision', 'fp16',
         '--d-model', '768', '--n-layers', '18', '--n-heads', '12', '--n-kv-heads', '3',
         '--batch-size', '16', '--seq-len', '512',
         '--data', 'data/pilot/train.bin']
r = subprocess.run(['python', 'src/train.py'] + PILOT +
                   ['--steps', '300', '--t-max', '6100', '--warmup-steps', '20',
                    '--ckpt-dir', 'checkpoints/pilot', '--log-every', '50',
                    '--ckpt-every', '300', '--val-every', '300'],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
assert 'device=cuda' in r.stdout, 'SILENT CPU FALLBACK?'
assert r.returncode == 0, 'probe failed:\n' + r.stderr[-2000:]
s_probe = json.load(open('checkpoints/pilot/last_summary.json'))
assert s_probe['precision'] == 'fp16' and s_probe['steps'] == 300
json.dump(s_probe, open('checkpoints/pilot/probe_summary.json', 'w'), indent=2)
print('probe summary saved to checkpoints/pilot/probe_summary.json (main run overwrites last_summary.json)')
print(f"PROBE OK — steady {s_probe['steady_tok_per_s']} tok/s, "
      f"peak {s_probe['peak_gpu_mem_mb']} MB allocated / {s_probe['peak_gpu_mem_reserved_mb']} MB reserved, "
      f"loss {s_probe['final_train_loss']:.4f}, skipped {s_probe['skipped_steps']}")

## Kill / resume test at 125M scale (separate directory)

A genuine interruption: a fresh 200-step run starts in the background,
is killed mid-run, and must resume from its last checkpoint to 200 steps
in a fresh process. This is the mechanism a multi-session full run depends
on — it is tested here at full model size, not assumed.

In [ ]:
import subprocess, json, os, time, glob

D = 'checkpoints/pilot_killtest'
base = ['python', 'src/train.py', '--device', 'cuda', '--precision', 'fp16',
        '--d-model', '768', '--n-layers', '18', '--n-heads', '12', '--n-kv-heads', '3',
        '--batch-size', '16', '--seq-len', '512',
        '--data', 'data/pilot/train.bin', '--ckpt-dir', D,
        '--log-every', '10', '--val-every', '1000']
p = subprocess.Popen(base + ['--steps', '200', '--t-max', '200', '--ckpt-every', '50'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(75)
p.kill()
out, _ = p.communicate()
print('killed run output tail:', out[-500:])
ckpts = sorted(glob.glob(os.path.join(D, 'ckpt_*.pt')))
assert ckpts, 'no checkpoint existed at kill time — kill test inconclusive'
kill_step = max(int(os.path.basename(c)[5:-3]) for c in ckpts)
print(f'killed with last checkpoint at step {kill_step}')
assert kill_step >= 50, 'kill happened before any checkpoint — rerun with longer sleep'
r = subprocess.run(base + ['--resume', '--steps', '200', '--t-max', '200', '--ckpt-every', '200'],
                   capture_output=True, text=True)
print(r.stdout[-800:])
assert r.returncode == 0, 'resume after kill failed:\n' + r.stderr[-2000:]
assert 'resumed scaler scale=' in r.stdout, 'scaler restore not logged'
s_kill = json.load(open(os.path.join(D, 'last_summary.json')))
assert s_kill['steps'] == 200 and s_kill['precision'] == 'fp16'
import math
assert math.isfinite(s_kill['final_train_loss']) and math.isfinite(s_kill['final_val_loss'])
kill_test = {'killed_at_checkpoint_step': kill_step, 'resumed_ok': True,
             'final_steps': 200, 'scaler_restored': True}
print('KILL/RESUME OK — interrupted at', kill_step, ', resumed to 200')

## Phase B — main pilot run (resume to 6,100 steps, ~50M tokens)

Resumes from the probe checkpoint to 6,100 steps (~one epoch over the
pilot slice). Checkpoints every 1,000 steps, val every 1,000, full log to
`pilot_run.log` (the loss curve is parsed from it in the results cell).
If this cell is interrupted (session death, quota pause), do NOT restart
it fresh — run the resume cell below it instead.

In [ ]:
import subprocess, json
r = subprocess.run(['python', 'src/train.py', '--device', 'cuda', '--precision', 'fp16',
                    '--d-model', '768', '--n-layers', '18', '--n-heads', '12', '--n-kv-heads', '3',
                    '--batch-size', '16', '--seq-len', '512',
                    '--data', 'data/pilot/train.bin',
                    '--resume', '--steps', '6100', '--t-max', '6100',
                    '--warmup-steps', '20',
                    '--ckpt-dir', 'checkpoints/pilot', '--log-every', '100',
                    '--ckpt-every', '1000', '--val-every', '1000',
                    '--log-file', 'checkpoints/pilot/pilot_run.log'],
                   capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, 'main pilot run failed:\n' + r.stderr[-2000:]
s_main = json.load(open('checkpoints/pilot/last_summary.json'))
assert s_main['steps'] == 6100 and s_main['precision'] == 'fp16'
print(f"MAIN OK — {s_main['tokens_seen']} tokens, loss {s_main['final_train_loss']:.4f}, "
      f"val {s_main['final_val_loss']:.4f}, skipped {s_main['skipped_steps']}, "
      f"scaler {s_main['scaler_final_scale']}")

## Resume cell — ONLY if Phase B was interrupted

Guarded: if the pilot summary already shows 6,100 steps, this cell skips
itself (re-running a completed run would crash on an empty step range).
Otherwise it resumes from the latest checkpoint to 6,100 steps.

In [ ]:
import json, os, subprocess
done = False
sp = 'checkpoints/pilot/last_summary.json'
if os.path.exists(sp):
    s = json.load(open(sp))
    done = (s.get('steps') == 6100 and s.get('precision') == 'fp16')
if done:
    print('pilot already complete at 6,100 steps — skipping resume')
else:
    r = subprocess.run(['python', 'src/train.py', '--device', 'cuda', '--precision', 'fp16',
                        '--d-model', '768', '--n-layers', '18', '--n-heads', '12', '--n-kv-heads', '3',
                        '--batch-size', '16', '--seq-len', '512',
                        '--data', 'data/pilot/train.bin',
                        '--resume', '--steps', '6100', '--t-max', '6100',
                        '--warmup-steps', '20',
                        '--ckpt-dir', 'checkpoints/pilot', '--log-every', '100',
                        '--ckpt-every', '1000', '--val-every', '1000',
                        '--log-file', 'checkpoints/pilot/pilot_run.log'],
                       capture_output=True, text=True)
    print(r.stdout[-1000:])
    assert r.returncode == 0, 'resume failed:\n' + r.stderr[-2000:]
    s_main = json.load(open(sp))
    assert s_main['steps'] == 6100
    print('RESUMED pilot complete at 6,100 steps')

## Eval — dev split ONLY (held-out split never touched)

The pilot checkpoint generates greedy completions for the 12 dev tasks
(byte-level: prompt bytes → generate → decode → truncate at the first
new top-level `def`). Empty-body and `return None` baselines run through
the identical harness on the same dev split for comparison. A score near
zero is EXPECTED at 50M tokens and validates the pipeline — it is not a
capability claim in either direction.

In [ ]:
import json, os, sys, glob, subprocess, math
sys.path.insert(0, 'src')
sys.path.insert(0, 'benchmarks')
import torch
from model import ForgeModel, ForgeConfig
from eval_harness import Problem, evaluate

# latest pilot checkpoint (rotation keeps last 2; latest == 6100)
ckpts = sorted(glob.glob('checkpoints/pilot/ckpt_*.pt'),
               key=lambda c: int(os.path.basename(c)[5:-3]))
assert ckpts, 'no pilot checkpoint found'
ck = torch.load(ckpts[-1], map_location='cuda', weights_only=False)
mc = ck['config']['model']
cfg = ForgeConfig(vocab_size=mc['vocab_size'], d_model=mc['d_model'], n_layers=mc['n_layers'],
                  n_heads=mc['n_heads'], n_kv_heads=mc['n_kv_heads'], max_seq_len=mc['max_seq_len'])
model = ForgeModel(cfg).to('cuda')
model.load_state_dict(ck['model'])
model.eval()
print('loaded', ckpts[-1], '| params:', cfg.param_count()['total'])

bench = json.load(open('benchmarks/forge_heldout_v1.json'))
dev = [t for t in bench['tasks'] if t['split'] == 'dev']
assert len(dev) == 12, f'dev split changed: {len(dev)} tasks'
completions = {}
for t in dev:
    prompt_b = t['prompt'].encode('utf-8')
    idx = torch.tensor([list(prompt_b)], dtype=torch.long, device='cuda')
    out = model.generate(idx, max_new=512, temperature=0.0)
    gen = bytes(out[0, len(prompt_b):].tolist()).decode('utf-8', errors='replace')
    cut = gen.find('\ndef ')
    if cut != -1:
        gen = gen[:cut]
    completions[t['id']] = gen
problems = [Problem(name=t['id'], prompt=t['prompt'], tests=t['tests'], entry_point=t['entry_point']) for t in dev]
res = evaluate(problems, completions, timeout=10)
print(f"pilot dev pass@1 = {res['passed']}/{res['total']} = {res['pass_at_1']:.3f}")
print('error funnel:', res['error_funnel'])
eval_dev = {'passed': res['passed'], 'total': res['total'], 'pass_at_1': res['pass_at_1'],
            'error_funnel': res['error_funnel'],
            'per_task': [{'id': r.problem, 'passed': r.passed, 'error_class': r.error_class} for r in res['results']]}
# identical baselines on dev
baselines = {}
for b in ('empty', 'none'):
    r = subprocess.run(['python', 'benchmarks/run_heldout.py', '--baseline', b, '--split', 'dev',
                        '--out', f'/kaggle/working/baseline_{b}_dev.json'],
                       capture_output=True, text=True)
    assert r.returncode == 0, f'baseline {b} failed:\n' + r.stderr[-1000:]
    bj = json.load(open(f'/kaggle/working/baseline_{b}_dev.json'))
    baselines[b] = {'passed': bj['passed'], 'total': bj['total'], 'pass_at_1': bj['pass_at_1']}
    print(f'baseline {b}: {bj["passed"]}/{bj["total"]}')
json.dump({'eval_dev': eval_dev, 'baselines': baselines},
          open('/kaggle/working/eval_dev.json', 'w'), indent=2)

## Results → JSON, scaling check, updated projection

**Methodology (explicit):**
- *Measured:* 125M steady tok/s (probe), peak memory, losses, skipped
  steps, scaler scale, kill/resume outcome, dev pass rate — from run
  summaries and the eval cell. No FLOPs-ratio extrapolation for the 125M
  projection anymore: hours = 2.5B tokens ÷ measured 125M steady tok/s.
- *Comparison (measured, not assumed):* 25M FP16 steady tok/s from EXP-006
  (39,356) vs 125M probe steady tok/s. The FLOPs-ratio prediction (same
  MFU at both sizes) is shown alongside the measured ratio — if they
  disagree, the measured number wins and the scaling assumption is revised.
- *Caveats that survive the pilot:* a 300-step probe and a 6,100-step run
  do not measure 305,000-step long-run behavior (full-corpus distribution
  shift, late-schedule LR phase, multi-session restart overhead). The
  projection below is an estimate with a band, and the budget adds
  overheads + failure contingency.

In [ ]:
import json, os, sys, re
sys.path.insert(0, 'src')
import torch
from model import ForgeConfig

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
vram = [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())]
corpus = json.load(open('/kaggle/working/corpus_stats.json'))
ev = json.load(open('/kaggle/working/eval_dev.json'))
s_probe = json.load(open('checkpoints/pilot/probe_summary.json'))
s_main = json.load(open('checkpoints/pilot/last_summary.json'))
assert s_probe['steps'] == 300 and s_main['steps'] == 6100
N125 = ForgeConfig(vocab_size=256, d_model=768, n_layers=18, n_heads=12, n_kv_heads=3, max_seq_len=513).param_count()['total']
fpt125 = 6 * N125 + 12 * 18 * 768 * 512
fpt25 = 6 * 25305600 + 12 * 8 * 512 * 512
# scaling check: FLOPs-ratio prediction (same MFU as 25M) vs MEASURED 125M steady tok/s
tps_25m = 39355.8  # EXP-006 measured FP16 steady tok/s at 25M
predicted_125m = tps_25m * fpt25 / fpt125
measured_125m = s_probe['steady_tok_per_s']
# updated projection from MEASURED 125M throughput (no FLOPs-ratio extrapolation)
hours_base = 2.5e9 / measured_125m / 3600
# loss curve: parse step/loss lines from the main run log (sampled)
curve = []
logp = 'checkpoints/pilot/pilot_run.log'
if os.path.exists(logp):
    for line in open(logp):
        m = re.search(r'step (\d+)/6100 loss=([0-9.]+)', line)
        if m:
            curve.append({'step': int(m.group(1)), 'loss': float(m.group(2))})
        m2 = re.search(r'val_loss=([0-9.]+)', line)
        if m2 and curve:
            curve[-1]['val_loss'] = float(m2.group(1))
curve = curve[::max(1, len(curve) // 30)] if curve else []
out = {
    'notebook': 'forge_m7_pilot',
    'pinned_commit': FORGE_COMMIT,
    'actual_head': globals().get('actual'),
    'hardware': {'gpu_names': gpu_names, 'gpu_count': len(gpu_names),
                 'vram_total_mb': [round(v / 1e6, 1) for v in vram],
                 'torch': torch.__version__, 'cuda_torch': torch.version.cuda,
                 'python': sys.version.split()[0]},
    'config': {'model': '125M-class', 'params_exact': N125, 'd_model': 768, 'n_layers': 18,
               'n_heads': 12, 'n_kv_heads': 3, 'precision': 'fp16', 'batch': 16, 'seq': 512,
               'steps': 6100, 't_max': 6100, 'optimizer': 'AdamW', 'lr': 3e-4, 'schedule': 'cosine'},
    'corpus': corpus,
    'probe_300step': s_probe,
    'main_run': s_main,
    'kill_test': kill_test,
    'loss_curve_sampled': curve,
    'eval_dev': ev['eval_dev'],
    'baselines_dev': ev['baselines'],
    'scaling_measured_vs_predicted': {
        'steady_tok_per_s_25M_fp16_exp006': tps_25m,
        'steady_tok_per_s_125M_measured': measured_125m,
        'steady_tok_per_s_125M_predicted_same_mfu': round(predicted_125m, 1),
        'measured_over_predicted': round(measured_125m / predicted_125m, 3),
        'note': 'predicted assumes identical MFU at 25M and 125M; measured wins if they disagree',
    },
    'projection_125m_2p5B_from_measured': {
        'hours_base': round(hours_base, 1),
        'hours_with_overheads_15pct': round(hours_base * 1.15, 1),
        'hours_planning_50pct': round(hours_base * 1.5, 1),
        'method': '2.5e9 tokens / measured 125M steady tok/s (no FLOPs-ratio extrapolation)',
        'caveat': 'short-run measurement; long-run (305k-step) effects unmeasured',
    },
    'status': 'ok',
}
json.dump(out, open('/kaggle/working/pilot_results.json', 'w'), indent=2, sort_keys=True)
print('=== PILOT VERDICT ===')
print(f"probe 125M steady: {measured_125m} tok/s (25M was {tps_25m}; same-MFU prediction {predicted_125m:.0f}; ratio {measured_125m / predicted_125m:.3f})")
print(f"probe peak: {s_probe['peak_gpu_mem_mb']} MB allocated / {s_probe['peak_gpu_mem_reserved_mb']} MB reserved")
print(f"main: {s_main['tokens_seen']} tokens, loss {s_main['final_train_loss']:.4f}, val {s_main['final_val_loss']:.4f}")
print(f"skipped {s_main['skipped_steps']}, nonfinite losses {s_main['nonfinite_losses']}, grads {s_main['nonfinite_grads']}, scaler {s_main['scaler_final_scale']}")
print(f"updated 125M/2.5B projection: {hours_base:.1f}h base, {hours_base * 1.15:.1f}h +overheads, {hours_base * 1.5:.1f}h planning")
print(f"dev eval: {ev['eval_dev']['passed']}/{ev['eval_dev']['total']} | empty {ev['baselines']['empty']['passed']}/{ev['baselines']['empty']['total']} | none {ev['baselines']['none']['passed']}/{ev['baselines']['none']['total']}")
print('wrote /kaggle/working/pilot_results.json')

## After the run

1. **Download** `pilot_results.json`, `corpus_stats.json`, and
   `baseline_{empty,none}_dev.json` from the notebook output.
2. Paste the pilot JSON back to Khushi — the EXP-007 final recommendation
   (proceed / another pilot / redesign) is made from measured numbers,
   including the updated 125M projection computed from the probe's
   steady tok/s (already inside the JSON).
3. **The full 125M / 2.5B run does NOT start from this notebook.** It needs
   a separate approval after the pilot evidence is reviewed.

**Troubleshooting:**
- `CUDA out of memory` on the 125M model → report peak memory from the
  probe cell; do not silently shrink the batch (it changes the experiment).
- Probe loss NaN / most steps skipped → AMP instability at 125M scale;
  that is a FAIL signal for the pilot, not something to tune away silently.
- Session death mid-run → run the guarded resume cell; the run continues
  from the latest 1,000-step checkpoint.